# Atelier AI — Fashion Recommendation Analysis

This notebook documents the development and evaluation of a fashion
recommendation system for client advisors.

The task is to recommend 12 distinct products per customer based on
purchase likelihood over the following seven days, using information
available before the prediction cutoff.

The current version establishes the notebook execution environment.
Data exploration and modelling will be added incrementally.

## 1. Environment

The notebook uses the project's Python environment and imports the
`atelier_ai` package installed from the repository's `src` directory.

Dependencies are declared in `pyproject.toml` and resolved in `uv.lock`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

import atelier_ai

print("Python version:", sys.version.split()[0])
print("Python executable:", sys.executable)
print("Package location:", Path(atelier_ai.__file__).resolve())
print("pandas version:", pd.__version__)

Python version: 3.12.14
Python executable: /Users/oumar/Developer/portfolio/atelier-ai-project/.venv/bin/python
Package location: /Users/oumar/Developer/portfolio/atelier-ai-project/src/atelier_ai/__init__.py
pandas version: 3.0.6


## 2. Data Source and Scope

The project uses the H&M Personalized Fashion Recommendations dataset.

The initial scope includes purchase transactions, article metadata and
customer metadata. Images are deferred.

Source documentation and handling decisions are recorded in the
[data card](../docs/data_card.md). Acquired files are identified in the
[data manifest](../docs/data_manifest.json).

## 3. Data Loading

### 3.1. File Locations and Preview Parameters

The initial loading check reads the first 1,000 records from each file.
These previews are not random samples and will not be used to infer
population-level patterns.

File locations are resolved from the project's editable package.

In [2]:
PROJECT_ROOT = Path(atelier_ai.__file__).resolve().parents[2]
RAW_DATA_DIRECTORY = PROJECT_ROOT / "data" / "raw" / "hm"

PREVIEW_ROWS = 1_000

SOURCE_FILENAMES = (
    "articles.csv",
    "customers.csv",
    "transactions_train.csv",
)

for filename in SOURCE_FILENAMES:
    path = RAW_DATA_DIRECTORY / filename
    if not path.is_file():
        raise FileNotFoundError(f"Required source file not found: {path}")

print("Raw data directory found.")

Raw data directory found.


### 3.2. Limited Reads

All fields are initially read as strings to preserve their lexical
representation, including identifiers with leading zeros.

Automatic missing-value recognition is disabled for this inspection.
Empty fields therefore remain empty strings. Numerical, categorical and
date conversions will be defined after inspection.

In [3]:
previews = {}

for filename in SOURCE_FILENAMES:
    dataset_name = Path(filename).stem

    previews[dataset_name] = pd.read_csv(
        RAW_DATA_DIRECTORY / filename,
        nrows=PREVIEW_ROWS,
        dtype="string",
        keep_default_na=False,
        encoding="utf-8",
    )

### 3.3. Loading Summary

The summary reports loaded dimensions and estimated DataFrame memory
usage for each preview. Loaded row counts are not full-file row counts.

Published outputs contain structural diagnostics rather than source
records.

In [4]:
summary_rows = []

for dataset_name, frame in previews.items():
    summary_rows.append(
        {
            "dataset": dataset_name,
            "rows_loaded": len(frame),
            "columns_loaded": frame.shape[1],
            "memory_mib": (
                frame.memory_usage(index=True, deep=True).sum() / 1024**2
            ),
        }
    )

loading_summary = pd.DataFrame(summary_rows)
loading_summary.round({"memory_mib": 3})

,dataset,rows_loaded,columns_loaded,memory_mib
0,articles,1000,25,1.440
1,customers,1000,7,0.464
2,transactions_train,1000,5,0.334
